In [ ]:
import pandas as pd
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# transaction = pd.read_json('transactions.txt',lines=True)

In [ ]:
# transaction.to_csv('transaction.csv',sep='|',index=False)

In [ ]:
import pandas as pd
import json
import numpy as np
from sklearn import metrics

import pandas
import random

# filename = 'transaction.csv'
# n = sum(1 for line in open(filename)) - 1 #number of records in file (excludes header)
# s = 10000 #desired sample size
# skip = sorted(random.sample(range(1,n+1),n-s)) #the 0-indexed header will not be included in the skip list
# transaction = pandas.read_csv(filename,sep='|', skiprows=skip)

transaction = pd.read_csv('transaction.csv',sep='|')

In [ ]:
transaction.dtypes

In [ ]:
transaction.describe()

In [ ]:
transaction.isnull().sum()

In [ ]:
len(transaction[transaction['isFraud'] == 1])

In [ ]:
transaction = transaction.sample(frac=1)

# amount of fraud classes 492 rows.
fraud_df = transaction.loc[transaction['isFraud'] == 1]
non_fraud_df = transaction.loc[transaction['isFraud'] == 0][:len(fraud_df)]

normal_distributed_df = pd.concat([fraud_df, non_fraud_df])

# Shuffle dataframe rows
transaction = normal_distributed_df.sample(frac=1, random_state=42)

transaction.head()

In [ ]:
print('Distribution of the Classes in the subsample dataset')
print(transaction['isFraud'].value_counts()/len(transaction))

sns.countplot('isFraud', data=transaction)
plt.title('Equally Distributed Classes', fontsize=14)
plt.show()

In [ ]:
transaction.columns

In [ ]:
binary_col = ['isFraud','cardPresent','expirationDateKeyInMatch']
for col in binary_col:
    transaction[col] = transaction[col].apply(lambda x: 1 if x == True else 0)

In [ ]:
column_dropped = ['customerId','accountNumber','echoBuffer',
                  'accountNumber','merchantCity','merchantState','posOnPremises','recurringAuthInd']
# char_column = ['acqCountry','posConditionCode','posEntryMode','transactionType',
#                'merchantZip','merchantName','merchantCategoryCode','merchantCountryCode']
char_column = ['acqCountry','posConditionCode','posEntryMode','transactionType',
               'merchantZip','merchantCategoryCode','merchantCountryCode']
imputed_column = ['acqCountry','merchantCountryCode','posConditionCode','posEntryMode','transactionType']
transaction = transaction.drop(columns=column_dropped)
transaction[char_column] = transaction[char_column].astype(str)

In [ ]:
char_column = ['acqCountry','posConditionCode','posEntryMode','transactionType',
               'merchantZip','merchantName','merchantCategoryCode','merchantCountryCode']
for i in char_column:
    print('{} : {}'.format(i,transaction[i].nunique()))

In [ ]:
from sklearn.impute import SimpleImputer

imp_mode = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
imp_mode.fit(transaction[imputed_column])
transaction[imputed_column] = imp_mode.transform(transaction[imputed_column])

In [ ]:
transaction.isnull().sum()

In [ ]:
transaction['acqCountry'].value_counts()

In [ ]:
#Change account open date to days since register
transaction['daysSinceRegister'] = transaction['accountOpenDate'].\
apply(lambda x: pd.to_datetime('today') - pd.to_datetime(x)).apply(lambda x: x.days)

##Change account open date to days since register
transaction['daysToExpiration'] = transaction['currentExpDate'].\
apply(lambda x: pd.to_datetime('today') - pd.to_datetime(x)).apply(lambda x: x.days)

##Change account open date to days since register
transaction['daySinceLastAddressChange'] = transaction['dateOfLastAddressChange'].\
apply(lambda x: pd.to_datetime('today') - pd.to_datetime(x)).apply(lambda x: x.days)

In [ ]:
transaction = transaction.drop(columns=['accountOpenDate'])
transaction = transaction.drop(columns=['currentExpDate'])
transaction = transaction.drop(columns=['dateOfLastAddressChange'])

In [ ]:
transaction['trx_hour'] = transaction['transactionDateTime'].apply(lambda x: pd.to_datetime(x).hour)
transaction['trx_day_of_week'] = transaction['transactionDateTime'].apply(lambda x: pd.to_datetime(x).dayofweek)
transaction['trx_day'] = transaction['transactionDateTime'].apply(lambda x: pd.to_datetime(x).day)
transaction['trx_month'] = transaction['transactionDateTime'].apply(lambda x: pd.to_datetime(x).month)

In [ ]:
transaction = transaction.drop(columns=['transactionDateTime'])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(categories='auto').fit(transaction[char_column])
transformed_data = encoder.transform(transaction[char_column]).toarray()
transformed_data = pd.DataFrame(transformed_data,columns=encoder.get_feature_names(char_column))
transaction = transaction.reset_index()
transaction = pd.concat([transaction, transformed_data], axis=1)
transaction = transaction.drop(columns=char_column)

In [ ]:
y = transaction['isFraud']
transaction = transaction.drop(columns=['isFraud'])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(transaction, y, test_size=0.2, random_state=42)

In [ ]:
transaction.head()

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import SVC
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier()
# model = SVC()

model.fit(X_train,y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
lw = 2
plt.plot(fpr, tpr, color='darkorange',
         lw=lw, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=lw, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic example')
plt.legend(loc="lower right")
plt.show()